# Reproducing LoRA on GLUE SST-2

Enable Kaggle Internet to clone the repository and download the model and dataset. Run the smoke test before the full experiments. No results are pre-filled.

In [ ]:
# Kaggle bootstrap: clone a reproducible source and install dependencies.
from pathlib import Path
import os, subprocess, sys
REPOSITORY_URL = 'https://github.com/ShreshthaJha6/LoRA_Implementation.git'
WORKING_DIR = Path(os.environ.get('KAGGLE_WORKING_DIR', '/kaggle/working'))
CHECKOUT_DIR = WORKING_DIR / 'LoRA_Implementation'
PROJECT_ROOT = CHECKOUT_DIR / 'lora-reproduction'
if not PROJECT_ROOT.is_dir():
    subprocess.check_call(['git', 'clone', REPOSITORY_URL, str(CHECKOUT_DIR)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')])
HF_HOME = WORKING_DIR / '.cache' / 'huggingface'
HF_DATASETS_CACHE = HF_HOME / 'datasets'
HF_HOME.mkdir(parents=True, exist_ok=True)
HF_DATASETS_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HF_DATASETS_CACHE'] = str(HF_DATASETS_CACHE)
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')
print(f'Hugging Face cache: {HF_HOME}')

In [ ]:
import torch
from src.config import default_experiments
from src.data import load_sst2, make_dataloaders
from src.model import build_model, build_tokenizer
from src.train import append_result, get_device, set_seed, train_experiment
from src.visualize import save_comparison_figure
device = get_device()
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'PyTorch CUDA build: {torch.version.cuda}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}; GPU count: {torch.cuda.device_count()}')
else:
    print('CUDA is unavailable; using CPU.')
print(f'Active device: {device}')

## Smoke test (run this first)

This separate path uses 32 SST-2 examples, LoRA rank 4, one epoch, and three optimization steps. It does not write experiment results.

In [ ]:
from dataclasses import replace
smoke_config = replace(default_experiments()[1], name='smoke_lora_r4', train_batch_size=8, eval_batch_size=8, num_epochs=1, max_train_steps=3)
set_seed(smoke_config.seed)
smoke_tokenizer = build_tokenizer(smoke_config)
smoke_train, smoke_validation = load_sst2(smoke_tokenizer, smoke_config)
smoke_train = smoke_train.select(range(min(32, len(smoke_train))))
smoke_validation = smoke_validation.select(range(min(32, len(smoke_validation))))
smoke_loaders = make_dataloaders(smoke_train, smoke_validation, smoke_tokenizer, smoke_config)
smoke_model = build_model(smoke_config)
smoke_result = train_experiment(smoke_model, *smoke_loaders, smoke_config, device)
assert {'validation_loss', 'accuracy', 'f1', 'trainable_parameters'} <= smoke_result.keys()
print(smoke_result)
del smoke_model
if torch.cuda.is_available(): torch.cuda.empty_cache()

## Full experiments (run only after smoke test succeeds)

Runs E0, E1 (rank 4), and E2 (rank 8), then appends only measured metrics to the CSV.

In [ ]:
experiments = default_experiments()
for experiment in experiments: print(f'Configured: {experiment.name} | rank: {experiment.lora_rank} | seed: {experiment.seed}')
set_seed(experiments[0].seed)
tokenizer = build_tokenizer(experiments[0])
train_dataset, validation_dataset = load_sst2(tokenizer, experiments[0])
results_path = PROJECT_ROOT / 'results' / 'experiment_results.csv'
all_results = []
for config in experiments:
    print(f'\nRunning {config.name}...')
    set_seed(config.seed)
    train_loader, validation_loader = make_dataloaders(train_dataset, validation_dataset, tokenizer, config)
    model = build_model(config)
    result = train_experiment(model, train_loader, validation_loader, config, device)
    append_result(result, results_path)
    all_results.append(result)
    print(result)
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
all_results

In [ ]:
# Run only after the full experiments.
figure_path = save_comparison_figure(results_path, PROJECT_ROOT / 'results' / 'figures')
print(f'Saved figure: {figure_path}')